In [1]:
import pandas as pd
import numpy as np

In [2]:
! pip install mysql-connector-python

In [3]:
import mysql.connector
print("done")

done


In [4]:
conn = mysql.connector.connect(
    host = 'localhost',
    user = 'root',
    password = 'Jomix#11',
    database = 'cart2insights',
    use_pure = True
 )

cursor = conn.cursor()
print("connection is made successfully") 


connection is made successfully


In [5]:
query = "use cart2insights"
cursor.execute(query)

In [6]:
query = """
SELECT
    order_id,
    SUM(price + freight_value) AS total_order_value
FROM order_items
GROUP BY order_id
"""

df_order_value = pd.read_sql(query, conn)

print(df_order_value.head())

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_10936\77899260.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_order_value = pd.read_sql(query, conn)


                           order_id  total_order_value
0  00010242fe8c5a6d1ba2dd792cb16214              72.19
1  00018f77f2f0320c557190d7a144bdd3             259.83
2  000229ec398224ef6ca0657da4fc703e             216.87
3  00024acbcdf0a6daa1e931b038114c75              25.78
4  00042b26cf59d7ce69dfabb4e55b4fd9             218.04


In [7]:
print(df_order_value.shape)
print(df_order_value.columns)

(98666, 2)
Index(['order_id', 'total_order_value'], dtype='object')


In [8]:
print(df_order_value.isnull().sum())

order_id             0
total_order_value    0
dtype: int64


In [9]:
query = """
SELECT
    order_id,
    order_purchase_Date,
    order_delivered_customer_date
FROM orders
"""

df_delivery = pd.read_sql(query, conn)

print(df_delivery.head())

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_10936\2213222886.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_delivery = pd.read_sql(query, conn)


                           order_id order_purchase_Date  \
0  00010242fe8c5a6d1ba2dd792cb16214          2017-09-13   
1  00018f77f2f0320c557190d7a144bdd3          2017-04-26   
2  000229ec398224ef6ca0657da4fc703e          2018-01-14   
3  00024acbcdf0a6daa1e931b038114c75          2018-08-08   
4  00042b26cf59d7ce69dfabb4e55b4fd9          2017-02-04   

  order_delivered_customer_date  
0                    2017-09-20  
1                    2017-05-12  
2                    2018-01-22  
3                    2018-08-14  
4                    2017-03-01  


In [10]:
print(df_delivery.shape)
print(df_delivery.isnull().sum())

(99441, 3)
order_id                            0
order_purchase_Date                 0
order_delivered_customer_date    2965
dtype: int64


In [12]:
df_6 = pd.read_csv("cleaned/olist_orders_dataset_cleaned.csv")

In [13]:
df_6["order_purchase_Date"] = pd.to_datetime(
    df_6["order_purchase_Date"]
)

df_6["order_delivered_customer_date"] = pd.to_datetime(
    df_6["order_delivered_customer_date"]
)

df_6["delivery_days"] = (
    df_6["order_delivered_customer_date"]
    - df_6["order_purchase_Date"]
).dt.days

print(df_6.head())

   Unnamed: 0                          order_id  \
0           0  e481f51cbdc54678b7cc49136f2d6af7   
1           1  53cdb2fc8bc7dce0b6741e2150273451   
2           2  47770eb9100c2d0c44946d9cf07ec65d   
3           3  949d5b44dbf5de918fe9c16f97b45f8a   
4           4  ad21c59c0840e6cb83a9ceb5573f8159   

                        customer_id order_status  \
0  9ef432eb6251297304e76186b10a928d    delivered   
1  b0830fb4747a6c6d20dea0b8c802d7ef    delivered   
2  41ce2a54c0b03bf3443c3d931a367089    delivered   
3  f88197465ea7920adcdbec7375364d82    delivered   
4  8ab97904e6daea8866dbdbc4fb7aad2c    delivered   

  order_delivered_customer_date order_estimated_delivery_date  \
0                    2017-10-10                    2017-10-18   
1                    2018-08-07                    2018-08-13   
2                    2018-08-17                    2018-09-04   
3                    2017-12-02                    2017-12-15   
4                    2018-02-16                    2018

In [14]:
print(df_6["delivery_days"].isnull().sum())
print(df_6["delivery_days"].describe())

2965
count    96476.000000
mean        12.497336
std          9.555460
min          0.000000
25%          7.000000
50%         10.000000
75%         16.000000
max        210.000000
Name: delivery_days, dtype: float64


In [15]:
df_6["order_estimated_delivery_date"] = pd.to_datetime(
    df_6["order_estimated_delivery_date"]
)

df_6["delivery_delay"] = (
    df_6["order_delivered_customer_date"]
    - df_6["order_estimated_delivery_date"]
).dt.days

print(df_6[[
    "order_id",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "delivery_delay"
]].head())

                           order_id order_delivered_customer_date  \
0  e481f51cbdc54678b7cc49136f2d6af7                    2017-10-10   
1  53cdb2fc8bc7dce0b6741e2150273451                    2018-08-07   
2  47770eb9100c2d0c44946d9cf07ec65d                    2018-08-17   
3  949d5b44dbf5de918fe9c16f97b45f8a                    2017-12-02   
4  ad21c59c0840e6cb83a9ceb5573f8159                    2018-02-16   

  order_estimated_delivery_date  delivery_delay  
0                    2017-10-18            -8.0  
1                    2018-08-13            -6.0  
2                    2018-09-04           -18.0  
3                    2017-12-15           -13.0  
4                    2018-02-26           -10.0  


In [16]:
print(df_6["delivery_delay"].isnull().sum())
print(df_6["delivery_delay"].describe())

2965
count    96476.000000
mean       -11.876881
std         10.183854
min       -147.000000
25%        -17.000000
50%        -12.000000
75%         -7.000000
max        188.000000
Name: delivery_delay, dtype: float64


In [40]:
df_6.head()

,Unnamed: 0,order_id,customer_id,order_status,order_delivered_customer_date,order_estimated_delivery_date,order_purchase_Time,order_purchase_Date,order_approved_at_Time,order_approved_at_Date,order_delivered_customer_time,delivery_days,delivery_delay,total_order_value
0,0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-10,2017-10-18,10:56:33,2017-10-02,11:07:15,2017-10-02,00:00:00,8.0,-8.0,38.71
1,1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-08-07,2018-08-13,20:41:37,2018-07-24,03:24:27,2018-07-26,00:00:00,14.0,-6.0,141.46
2,2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-17,2018-09-04,08:38:49,2018-08-08,08:55:23,2018-08-08,00:00:00,9.0,-18.0,179.12
3,3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-12-02,2017-12-15,19:28:06,2017-11-18,19:45:59,2017-11-18,00:00:00,14.0,-13.0,72.20
4,4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-16,2018-02-26,21:18:39,2018-02-13,22:20:29,2018-02-13,00:00:00,3.0,-10.0,28.62


In [17]:
query = """
SELECT
    c.customer_unique_id,
    COUNT(DISTINCT o.order_id) AS customer_order_count
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
GROUP BY c.customer_unique_id
"""

df_customer_orders = pd.read_sql(query, conn)

print(df_customer_orders.head())

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_10936\3338292880.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_customer_orders = pd.read_sql(query, conn)


                 customer_unique_id  customer_order_count
0  0000366f3b9a7992bf8c76cfdf3221e2                     1
1  0000b849f77a49e4a4ce2b2a4ca5be3f                     1
2  0000f46a3911fa3c0805444483337064                     1
3  0000f6ccb0745a6a4b88665a16c9f078                     1
4  0004aac84e0df4da2b147fca70cf8255                     1


In [18]:
print(df_customer_orders.shape)
print(df_customer_orders.isnull().sum())

(96096, 2)
customer_unique_id      0
customer_order_count    0
dtype: int64


In [42]:
df = pd.read_csv("cleaned/olist_customers_dataset_cleaned.csv")

In [43]:
df.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [19]:
query = """
SELECT
    c.customer_unique_id,
    SUM(oi.price + oi.freight_value) AS customer_total_spending
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY c.customer_unique_id
"""

df_customer_spending = pd.read_sql(query, conn)

print(df_customer_spending.head())

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_10936\2820109364.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_customer_spending = pd.read_sql(query, conn)


                 customer_unique_id  customer_total_spending
0  0000366f3b9a7992bf8c76cfdf3221e2                   141.90
1  0000b849f77a49e4a4ce2b2a4ca5be3f                    27.19
2  0000f46a3911fa3c0805444483337064                    86.22
3  0000f6ccb0745a6a4b88665a16c9f078                    43.62
4  0004aac84e0df4da2b147fca70cf8255                   196.89


In [20]:
print(df_customer_spending.shape)
print(df_customer_spending.isnull().sum())

(95420, 2)
customer_unique_id         0
customer_total_spending    0
dtype: int64


In [21]:
df_customer = pd.merge(
    df_customer_orders,
    df_customer_spending,
    on="customer_unique_id"
)

print(df_customer.head())

                 customer_unique_id  customer_order_count  \
0  0000366f3b9a7992bf8c76cfdf3221e2                     1   
1  0000b849f77a49e4a4ce2b2a4ca5be3f                     1   
2  0000f46a3911fa3c0805444483337064                     1   
3  0000f6ccb0745a6a4b88665a16c9f078                     1   
4  0004aac84e0df4da2b147fca70cf8255                     1   

   customer_total_spending  
0                   141.90  
1                    27.19  
2                    86.22  
3                    43.62  
4                   196.89  


In [22]:
df_customer["average_order_value"] = (
    df_customer["customer_total_spending"]
    / df_customer["customer_order_count"]
)

print(df_customer.head())

                 customer_unique_id  customer_order_count  \
0  0000366f3b9a7992bf8c76cfdf3221e2                     1   
1  0000b849f77a49e4a4ce2b2a4ca5be3f                     1   
2  0000f46a3911fa3c0805444483337064                     1   
3  0000f6ccb0745a6a4b88665a16c9f078                     1   
4  0004aac84e0df4da2b147fca70cf8255                     1   

   customer_total_spending  average_order_value  
0                   141.90               141.90  
1                    27.19                27.19  
2                    86.22                86.22  
3                    43.62                43.62  
4                   196.89               196.89  


In [23]:
df_customer["average_order_value"] = (
    df_customer["customer_total_spending"]
    / df_customer["customer_order_count"]
)

print(df_customer.head())

                 customer_unique_id  customer_order_count  \
0  0000366f3b9a7992bf8c76cfdf3221e2                     1   
1  0000b849f77a49e4a4ce2b2a4ca5be3f                     1   
2  0000f46a3911fa3c0805444483337064                     1   
3  0000f6ccb0745a6a4b88665a16c9f078                     1   
4  0004aac84e0df4da2b147fca70cf8255                     1   

   customer_total_spending  average_order_value  
0                   141.90               141.90  
1                    27.19                27.19  
2                    86.22                86.22  
3                    43.62                43.62  
4                   196.89               196.89  


In [44]:
df.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [24]:
query = """
SELECT
    seller_id,
    SUM(price) AS seller_revenue
FROM order_items
GROUP BY seller_id
"""

df_seller_revenue = pd.read_sql(query, conn)

print(df_seller_revenue.head())

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_10936\102255800.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_seller_revenue = pd.read_sql(query, conn)


                          seller_id  seller_revenue
0  0015a82c2db000af6aaaf3ae2ecb0532         2685.00
1  001cca7ae9ae17fb1caed9dfb1094831        25080.03
2  001e6ad469a905060d959994f1b41e4f          250.00
3  002100f778ceb8431b7a1020ff7ab48f         1234.50
4  003554e2dce176b5555353e4f3555ac8          120.00


In [25]:
print(df_seller_revenue.shape)
print(df_seller_revenue.isnull().sum())

(3095, 2)
seller_id         0
seller_revenue    0
dtype: int64


In [26]:
query = """
SELECT
    seller_id,
    COUNT(DISTINCT order_id) AS seller_order_count
FROM order_items
GROUP BY seller_id
"""

df_seller_orders = pd.read_sql(query, conn)

print(df_seller_orders.head())

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_10936\648103984.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_seller_orders = pd.read_sql(query, conn)


                          seller_id  seller_order_count
0  0015a82c2db000af6aaaf3ae2ecb0532                   3
1  001cca7ae9ae17fb1caed9dfb1094831                 200
2  001e6ad469a905060d959994f1b41e4f                   1
3  002100f778ceb8431b7a1020ff7ab48f                  51
4  003554e2dce176b5555353e4f3555ac8                   1


In [27]:
print(df_seller_orders.shape)
print(df_seller_orders.isnull().sum())

(3095, 2)
seller_id             0
seller_order_count    0
dtype: int64


In [28]:
df_customer["repeat_customer"] = np.where(
    df_customer["customer_order_count"] > 1,
    1,
    0
)

print(df_customer.head())

                 customer_unique_id  customer_order_count  \
0  0000366f3b9a7992bf8c76cfdf3221e2                     1   
1  0000b849f77a49e4a4ce2b2a4ca5be3f                     1   
2  0000f46a3911fa3c0805444483337064                     1   
3  0000f6ccb0745a6a4b88665a16c9f078                     1   
4  0004aac84e0df4da2b147fca70cf8255                     1   

   customer_total_spending  average_order_value  repeat_customer  
0                   141.90               141.90                0  
1                    27.19                27.19                0  
2                    86.22                86.22                0  
3                    43.62                43.62                0  
4                   196.89               196.89                0  


In [29]:
print(df_customer["repeat_customer"].value_counts())
print(df_customer.isnull().sum())

repeat_customer
0    92431
1     2989
Name: count, dtype: int64
customer_unique_id         0
customer_order_count       0
customer_total_spending    0
average_order_value        0
repeat_customer            0
dtype: int64


In [30]:
print(df_6.columns)
print(df_6.shape)

Index(['Unnamed: 0', 'order_id', 'customer_id', 'order_status',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'order_purchase_Time', 'order_purchase_Date', 'order_approved_at_Time',
       'order_approved_at_Date', 'order_delivered_customer_time',
       'delivery_days', 'delivery_delay'],
      dtype='object')
(99441, 13)


In [32]:
df_3 = pd.read_csv("cleaned/olist_order_items_dataset_cleaned.csv")

In [33]:
df_order_value = df_3.groupby("order_id").apply(
    lambda x: (x["price"] + x["freight_value"]).sum()
).reset_index(name="total_order_value")

print(df_order_value.head())

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_10936\3668165592.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_order_value = df_3.groupby("order_id").apply(


                           order_id  total_order_value
0  00010242fe8c5a6d1ba2dd792cb16214              72.19
1  00018f77f2f0320c557190d7a144bdd3             259.83
2  000229ec398224ef6ca0657da4fc703e             216.87
3  00024acbcdf0a6daa1e931b038114c75              25.78
4  00042b26cf59d7ce69dfabb4e55b4fd9             218.04


In [34]:
df_6 = pd.merge(
    df_6,
    df_order_value,
    on="order_id",
    how="left"
)

print(df_6.head())

   Unnamed: 0                          order_id  \
0           0  e481f51cbdc54678b7cc49136f2d6af7   
1           1  53cdb2fc8bc7dce0b6741e2150273451   
2           2  47770eb9100c2d0c44946d9cf07ec65d   
3           3  949d5b44dbf5de918fe9c16f97b45f8a   
4           4  ad21c59c0840e6cb83a9ceb5573f8159   

                        customer_id order_status  \
0  9ef432eb6251297304e76186b10a928d    delivered   
1  b0830fb4747a6c6d20dea0b8c802d7ef    delivered   
2  41ce2a54c0b03bf3443c3d931a367089    delivered   
3  f88197465ea7920adcdbec7375364d82    delivered   
4  8ab97904e6daea8866dbdbc4fb7aad2c    delivered   

  order_delivered_customer_date order_estimated_delivery_date  \
0                    2017-10-10                    2017-10-18   
1                    2018-08-07                    2018-08-13   
2                    2018-08-17                    2018-09-04   
3                    2017-12-02                    2017-12-15   
4                    2018-02-16                    2018

In [35]:
print(df_6.shape)
print(df_6["total_order_value"].isnull().sum())

(99441, 14)
775


In [36]:
print(df_6[df_6["total_order_value"].isnull()][
    ["order_id", "order_status", "customer_id"]
].head(20))

                              order_id order_status  \
266   8e24261a7e58791d10cb1bf9da94df5c  unavailable   
586   c272bcd21c287498b4883c7512019702  unavailable   
687   37553832a3a89c9b2db59701c357ca67  unavailable   
737   d57e15fb07fd180f06ab3926b39edcd2  unavailable   
1130  00b1cb0320190ca0daa2c88b35206009     canceled   
1160  2f634e2cebf8c0283e7ef0989f77d217  unavailable   
1579  ee0db22a8e742b752914016708470ec8  unavailable   
1801  ed3efbd3a87bea76c2812c66a0b32219     canceled   
1826  6ad57aecbae806a7e9cc2cdb6b380711  unavailable   
1868  df8282afe61008dc26c6c31011474d02     canceled   
2029  8d4c637f1accf7a88a4555f02741e606     canceled   
2148  9f3458f0433d4cfa472f4682654b8d49  unavailable   
2161  7a9d4c7f9b068337875b95465330f2fc     canceled   
2296  f0df036e9be21fc238f131bbfcdde769  unavailable   
2316  86f0bcd9b426a2c1022ae317c7f27be6  unavailable   
2680  98fe01bc254b759c1f2c11d77c2891d5  unavailable   
2707  f56dc9034c3962e36d49fb73baa12800  unavailable   
2971  0130

In [37]:
print(df_6[df_6["total_order_value"].isnull()]["order_status"].value_counts())

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64


In [38]:
print(df_6["total_order_value"].describe())

count    98666.000000
mean       160.577638
std        220.466087
min          9.590000
25%         61.980000
50%        105.290000
75%        176.870000
max      13664.080000
Name: total_order_value, dtype: float64


In [39]:
print(df_6[["order_id", "total_order_value"]].head(10))

                           order_id  total_order_value
0  e481f51cbdc54678b7cc49136f2d6af7              38.71
1  53cdb2fc8bc7dce0b6741e2150273451             141.46
2  47770eb9100c2d0c44946d9cf07ec65d             179.12
3  949d5b44dbf5de918fe9c16f97b45f8a              72.20
4  ad21c59c0840e6cb83a9ceb5573f8159              28.62
5  a4591c265e18cb1dcee52889e2d8acc3             175.26
6  136cce7faa42fdb2cefd53fdc79a6098              65.95
7  6514b8ad8028c9f2cc2374ded245783f              75.16
8  76c6e866289321a7c93b82b54852dc33              35.95
9  e69bfb5eb88e0ed6a785585b27e16dbf             169.76


In [45]:
df_6.head()

,Unnamed: 0,order_id,customer_id,order_status,order_delivered_customer_date,order_estimated_delivery_date,order_purchase_Time,order_purchase_Date,order_approved_at_Time,order_approved_at_Date,order_delivered_customer_time,delivery_days,delivery_delay,total_order_value
0,0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-10,2017-10-18,10:56:33,2017-10-02,11:07:15,2017-10-02,00:00:00,8.0,-8.0,38.71
1,1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-08-07,2018-08-13,20:41:37,2018-07-24,03:24:27,2018-07-26,00:00:00,14.0,-6.0,141.46
2,2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-17,2018-09-04,08:38:49,2018-08-08,08:55:23,2018-08-08,00:00:00,9.0,-18.0,179.12
3,3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-12-02,2017-12-15,19:28:06,2017-11-18,19:45:59,2017-11-18,00:00:00,14.0,-13.0,72.20
4,4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-16,2018-02-26,21:18:39,2018-02-13,22:20:29,2018-02-13,00:00:00,3.0,-10.0,28.62


In [46]:
print(df_6.columns.tolist())

['Unnamed: 0', 'order_id', 'customer_id', 'order_status', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_purchase_Time', 'order_purchase_Date', 'order_approved_at_Time', 'order_approved_at_Date', 'order_delivered_customer_time', 'delivery_days', 'delivery_delay', 'total_order_value']


In [47]:
df_6.to_csv("orders_feature_engineered.csv", index=False)